# LEVER: 在线自适应序列学习框架

## 论文参考

- **标题**: LEVER: Online Adaptive Sequence Learning Framework
- **作者**: Zixuan Yuan
- **年份**: 2024
- **关键结果**: 夏普比率提升 +0.27

### 摘要

本文提出 LEVER 在线学习框架，模型在每个时间步接收新观测后立即更新预测参数。
使用指数加权更新机制，使模型能快速适应市场变化。
本 notebook 使用日线数据实现简化版在线学习策略。

> **⚠️ 教学演示声明**
> 
> 本 Notebook 为量化策略算法教学样例，回测结果包含**前视偏差 (Look-ahead Bias)**：
> - 训练标签使用了未来收益率（`shift(-N)`）
> - StandardScaler 等预处理可能在全量数据上拟合
> - 模型参数选择可能基于完整样本期
> 
> **所有回测收益率仅供学习参考，不代表策略的实际可期望表现，不可直接用于实盘交易。**

## 算法原理

### 在线自适应学习

**1. 基本思想**

传统离线模型训练一次后固定参数。在线学习在每个时间步 $t$ 更新:

$$w_{t+1} = w_t - \eta_t \cdot \nabla L(w_t, x_t, y_t)$$

**2. 指数加权在线学习**

LEVER 使用指数加权更新，近期观测权重更大:

$$\hat{y}_{t+1} = \sum_{i=1}^{t} \lambda^{t-i} w_i^\top x_i \bigg/ \sum_{i=1}^{t} \lambda^{t-i}$$

其中 $\lambda \in (0, 1)$ 是遗忘因子 (forgetting factor)。

**3. 递归最小二乘 (RLS) 实现**

$$K_t = \frac{P_{t-1} x_t}{\lambda + x_t^\top P_{t-1} x_t}$$
$$w_t = w_{t-1} + K_t (y_t - x_t^\top w_{t-1})$$
$$P_t = \frac{1}{\lambda}(P_{t-1} - K_t x_t^\top P_{t-1})$$

**4. 自适应优势**

- 无需重新训练: 参数持续更新
- 快速适应: 市场 regime 变化时自动调整
- 低延迟: 单步更新计算量极小

In [ ]:
# ============================================================
# Cell 3: 动画展示 - 在线学习实时适应
# ============================================================
import numpy as np
import plotly.graph_objects as go

np.random.seed(42)
n = 200

# 模拟: 前100步线性趋势, 后100步反转
x = np.linspace(0, 4*np.pi, n)
y_true = np.sin(x) + 0.5 * np.sin(2*x)
y_obs = y_true + np.random.randn(n) * 0.3

# 在线学习预测
preds_online = np.zeros(n)
errors_online = np.zeros(n)
alpha_online = 0.1  # 学习率

# 简单指数平滑预测
ema_pred = y_obs[0]
for t in range(n):
    preds_online[t] = ema_pred
    errors_online[t] = abs(y_obs[t] - ema_pred)
    ema_pred = (1 - alpha_online) * ema_pred + alpha_online * y_obs[t]

# 离线模型 (全局均值)
cumsum = np.cumsum(y_obs)
preds_offline = cumsum / np.arange(1, n+1)
errors_offline = np.abs(y_obs - preds_offline)

# 动画
frames = []
step = 4
for t in range(20, n, step):
    frames.append(go.Frame(
        data=[
            go.Scatter(x=list(range(t)), y=y_obs[:t].tolist(), mode='markers',
                       name='观测值', marker=dict(color='gray', size=3, opacity=0.5)),
            go.Scatter(x=list(range(t)), y=preds_online[:t].tolist(), mode='lines',
                       name='在线学习', line=dict(color='#F44336', width=2.5)),
            go.Scatter(x=list(range(t)), y=preds_offline[:t].tolist(), mode='lines',
                       name='离线模型', line=dict(color='#2196F3', width=2, dash='dash')),
            go.Scatter(x=list(range(t)),
                       y=pd.Series(errors_online[:t]).rolling(10).mean().tolist(),
                       mode='lines', name='在线误差(MA10)',
                       line=dict(color='#FF9800', width=1.5), yaxis='y2'),
        ],
        name=f't={t}'
    ))

import pandas as pd
fig = go.Figure(
    data=frames[0].data,
    frames=frames,
    layout=go.Layout(
        title=dict(text='在线学习 vs 离线模型: 实时适应能力对比'),
        xaxis_title='时间步', yaxis=dict(title='预测值'),
        yaxis2=dict(title='预测误差', overlaying='y', side='right', range=[0, 1]),
        height=500, width=950,
        updatemenus=[dict(type='buttons', buttons=[
            dict(label='\u25b6 播放', method='animate',
                 args=[None, {'frame': {'duration': 100}, 'transition': {'duration': 50}}]),
        ])],
        sliders=[dict(
            steps=[dict(args=[[f.name], {'frame': {'duration': 0}, 'mode': 'immediate'}],
                        label=f.name, method='animate') for f in frames],
        )],
    )
)
fig.show()

In [ ]:
# ============================================================
# Cell 4: 环境设置与导入
# ============================================================
import sys
import warnings
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings('ignore')
np.random.seed(42)

sys.path.insert(0, '..')
from shared.data_fetcher import get_etf_daily, get_index_daily
from shared.backtest_engine import Backtester
from shared.factors import momentum, volatility, rsi, macd, bollinger_bands
from shared.visualization import (
    plot_equity_curve, plot_drawdown, plot_metrics_table,
    plot_cumulative_comparison
)
from shared.constants import DEFAULT_START, DEFAULT_END, ETF_CSI300

print('所有模块导入成功')

In [ ]:
# ============================================================
# Cell 5: 数据获取
# ============================================================
try:
    df = get_etf_daily(ETF_CSI300, DEFAULT_START, DEFAULT_END)
    print(f'ETF 510300 数据: {len(df)} 条')
except Exception as e:
    print(f'数据获取异常: {e}, 使用模拟数据')
    dates = pd.bdate_range(DEFAULT_START, DEFAULT_END)
    n = len(dates)
    np.random.seed(42)
    price = 4.5 + np.cumsum(np.random.randn(n) * 0.015)
    price = np.maximum(price, 2.0)
    df = pd.DataFrame({
        'open': price * (1 + np.random.randn(n) * 0.003),
        'close': price,
        'high': price * (1 + np.abs(np.random.randn(n) * 0.008)),
        'low': price * (1 - np.abs(np.random.randn(n) * 0.008)),
        'volume': np.random.randint(50000, 2000000, n).astype(float),
    }, index=dates)

print(f'数据范围: {df.index[0].strftime("%Y-%m-%d")} ~ {df.index[-1].strftime("%Y-%m-%d")}')

In [ ]:
# ============================================================
# Cell 6: 因子工程 + 在线RLS学习器
# ============================================================

# 构建因子
features = pd.DataFrame(index=df.index)
mom = momentum(df['close'], [5, 10, 20])
features = pd.concat([features, mom], axis=1)
vol = volatility(df['close'], [5, 10])
features = pd.concat([features, vol], axis=1)
features['rsi'] = rsi(df['close'], 14)
macd_df = macd(df['close'])
features['macd_hist'] = macd_df['histogram']
bb = bollinger_bands(df['close'])
features['bb_pctb'] = bb['bb_pctb']
features['vol_chg'] = df['volume'].pct_change(5)

# 标签: 下一日收益率
forward_ret = df['close'].pct_change().shift(-1)

# 清洗
features = features.dropna()
forward_ret = forward_ret.reindex(features.index).dropna()
common_idx = features.index.intersection(forward_ret.index)
features = features.loc[common_idx]
forward_ret = forward_ret.loc[common_idx]

FEATURE_COLS = features.columns.tolist()
print(f'因子: {FEATURE_COLS}')
print(f'样本数: {len(features)}')


class OnlineRLS:
    """递归最小二乘 (RLS) 在线学习器"""

    def __init__(self, n_features, forgetting_factor=0.99, delta=1.0):
        self.n = n_features
        self.lam = forgetting_factor
        self.w = np.zeros(n_features)       # 权重向量
        self.P = np.eye(n_features) * delta  # 协方差矩阵

    def predict(self, x):
        return np.dot(self.w, x)

    def update(self, x, y):
        """接收新观测后更新参数"""
        # 卡尔曼增益
        Px = self.P @ x
        denom = self.lam + x @ Px
        K = Px / denom

        # 预测误差
        error = y - np.dot(self.w, x)

        # 更新权重
        self.w = self.w + K * error

        # 更新协方差
        self.P = (self.P - np.outer(K, x @ self.P)) / self.lam

        return error


print('RLS 在线学习器定义完成')

In [ ]:
# ============================================================
# Cell 7: 在线学习预测 + 滚动窗口离线对比
# ============================================================
from sklearn.linear_model import Ridge

X = features.values
y = forward_ret.values

# 标准化 (使用滚动统计)
scaler = StandardScaler()

# 在线学习 (RLS)
n_features = X.shape[1]
rls = OnlineRLS(n_features, forgetting_factor=0.99)

online_preds = []
online_errors = []
warmup = 60  # 预热期

# 用前 warmup 天标准化参考
scaler.fit(X[:warmup])

for t in range(len(X)):
    x_t = scaler.transform(X[t:t+1]).flatten()

    if t < warmup:
        online_preds.append(0)
        rls.update(x_t, y[t])
        online_errors.append(abs(y[t]))
    else:
        pred = rls.predict(x_t)
        online_preds.append(pred)
        error = rls.update(x_t, y[t])
        online_errors.append(abs(error))

    # 每100步更新一次scaler (防止数值漂移)
    if t > 0 and t % 100 == 0:
        start = max(0, t - 200)
        scaler.fit(X[start:t])

online_preds = np.array(online_preds)

# 离线滚动窗口模型 (对比)
OFFLINE_WINDOW = 120
offline_preds = np.zeros(len(X))

for t in range(OFFLINE_WINDOW, len(X)):
    X_train = X[t - OFFLINE_WINDOW:t]
    y_train = y[t - OFFLINE_WINDOW:t]
    local_scaler = StandardScaler()
    X_train_s = local_scaler.fit_transform(X_train)
    ridge = Ridge(alpha=1.0)
    ridge.fit(X_train_s, y_train)
    x_t_s = local_scaler.transform(X[t:t+1])
    offline_preds[t] = ridge.predict(x_t_s)[0]

# 预测方向准确率
test_start = max(warmup, OFFLINE_WINDOW)
online_dir_acc = ((np.sign(online_preds[test_start:]) == np.sign(y[test_start:])).mean())
offline_dir_acc = ((np.sign(offline_preds[test_start:]) == np.sign(y[test_start:])).mean())
print(f'在线 RLS 方向准确率: {online_dir_acc:.2%}')
print(f'离线 Ridge 方向准确率: {offline_dir_acc:.2%}')

In [ ]:
# ============================================================
# Cell 8: 信号生成与回测
# ============================================================

dates = features.index

# 在线策略信号
online_signals = pd.Series(0, index=dates)
online_signals[online_preds > 0] = 1
online_signals.iloc[:warmup] = 0

# 离线策略信号
offline_signals = pd.Series(0, index=dates)
offline_signals[offline_preds > 0] = 1
offline_signals.iloc[:OFFLINE_WINDOW] = 0

# 获取基准
try:
    bench = get_index_daily('sh000300', DEFAULT_START, DEFAULT_END)
    bench_prices = bench['close']
except:
    bench_prices = None

backtester = Backtester(initial_capital=1_000_000, t_plus_1=True)

online_result = backtester.run(df['close'], online_signals, bench_prices)
offline_result = backtester.run(df['close'], offline_signals, bench_prices)

print('=== LEVER 在线学习策略 ===')
for k, v in online_result['metrics'].items():
    print(f'  {k}: {v}')

print('\n=== 离线滚动窗口策略 ===')
for k, v in offline_result['metrics'].items():
    print(f'  {k}: {v}')

In [ ]:
# ============================================================
# Cell 9: 绩效可视化
# ============================================================
import matplotlib.pyplot as plt

# 1. 策略对比
strategies = {
    'LEVER (在线RLS)': online_result['returns'],
    '离线Ridge (120日窗口)': offline_result['returns'],
}
plot_cumulative_comparison(strategies, title='LEVER在线学习 vs 离线滚动窗口')

# 2. 收益曲线
bench_eq = None
if online_result.get('benchmark_returns') is not None:
    bench_eq = (1 + online_result['benchmark_returns']).cumprod() * online_result['equity_curve'].iloc[0]
plot_equity_curve(online_result['equity_curve'], bench_eq,
                  title='LEVER 在线学习 - 累计收益')

# 3. 回撤
plot_drawdown(online_result['equity_curve'], title='LEVER - 回撤')

# 4. 预测误差随时间衰减
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

error_ma = pd.Series(online_errors).rolling(30).mean()
ax1.plot(error_ma, color='#F44336', linewidth=1.5, label='在线学习误差(MA30)')
ax1.set_xlabel('时间步')
ax1.set_ylabel('绝对预测误差')
ax1.set_title('在线学习: 预测误差随时间下降')
ax1.legend()
ax1.grid(True, alpha=0.3)

# 滚动方向准确率
rolling_acc_online = pd.Series(
    (np.sign(online_preds) == np.sign(y)).astype(float)
).rolling(60).mean()
rolling_acc_offline = pd.Series(
    (np.sign(offline_preds) == np.sign(y)).astype(float)
).rolling(60).mean()

ax2.plot(rolling_acc_online.values, color='#F44336', label='在线RLS', linewidth=1.5)
ax2.plot(rolling_acc_offline.values, color='#2196F3', label='离线Ridge', linewidth=1.5)
ax2.axhline(y=0.5, color='gray', linestyle='--', alpha=0.5)
ax2.set_xlabel('时间步')
ax2.set_ylabel('方向准确率')
ax2.set_title('滚动方向准确率 (60日)')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# 5. 绩效表
plot_metrics_table(online_result['metrics'], title='LEVER 在线学习绩效指标')

## 结果讨论

### 策略表现

1. **在线适应**: RLS 模型无需重新训练，每步 O(d^2) 更新
2. **遗忘因子**: lambda=0.99 约等于看过去 100 天数据加权平均
3. **vs 离线**: 在线模型在 regime 切换时响应更快

### 局限性

- RLS 假设线性关系，无法捕捉非线性模式
- 遗忘因子需要调优 (太小遗忘太快，太大适应太慢)
- 日线数据限制了在线学习的更新频率优势

### 改进方向

- 使用核化 RLS (Kernel RLS) 处理非线性关系
- 自适应遗忘因子: 根据预测误差动态调整 lambda
- 在线深度学习: 使用增量训练的神经网络
- 使用更高频数据 (分钟线) 发挥在线学习优势